# Per-Market Fine-Tuning

Set MODEL_NAME and FOLD before running.

In [ ]:
MODEL_NAME = "thai_equity"  # one of: thai_equity, us_equity, crypto
FOLD = 0                    # 0, 1, or 2
MODE = "train"              # "train" or "holdout"
print(f"{MODEL_NAME} fold {FOLD} mode={MODE}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install yfinance pandas numpy
!git clone --depth 1 https://github.com/shiyu-coder/Kronos.git kronos_repo
!pip install -r kronos_repo/requirements.txt
import sys; sys.path.insert(0, 'kronos_repo')

In [ ]:
sys.path.insert(0, '/content/drive/MyDrive/kronos-th')
from kth.models.finetune import prepare_dataset, finetune_tokenizer, finetune_predictor, evaluate_model
from kth.data.universe import UNIVERSE

In [ ]:
MODEL_TICKERS = {
    "thai_equity": [t for t,_,_ in UNIVERSE["thai_equity"]],
    "us_equity":   [t for t,_,_ in UNIVERSE["us_equity"]],
    "crypto":      [t for t,_,_ in UNIVERSE["crypto"]],
}
tickers = MODEL_TICKERS[MODEL_NAME]
print(f"Tickers: {len(tickers)}")

In [ ]:
if MODE == "train":
    dataset = prepare_dataset(
        tickers=tickers,
        cache_dir="/content/drive/MyDrive/kronos-th/data/raw",
        fold=FOLD,
    )
    print(f"Train: {len(dataset["train"])}")
    print(f"Val:   {len(dataset["val"])}")
elif MODE == "holdout":
    dataset = prepare_dataset(
        tickers=tickers,
        cache_dir="/content/drive/MyDrive/kronos-th/data/raw",
        holdout_start_date="2025-01-01",
    )
print(f"Test:  {len(dataset["test"])}")


In [ ]:
# Cell 6: Tokenizer (DEPRECATED)
print("Tokenizer fine-tuning is DEPRECATED — Kronos has no fit() method.")
print("Using pre-trained tokenizer instead.")
tok_path = "NeoQuasar/Kronos-Tokenizer-base"
print(f"Tokenizer: {tok_path}")


In [ ]:
# Cell 7: Predictor
if MODE == "train":
    import subprocess
    subprocess.run([
        "python", "scripts/train_per_market.py", MODEL_NAME, str(FOLD)
    ])
    pred_path = f"/content/drive/MyDrive/kronos-th/checkpoints/{MODEL_NAME}/fold{FOLD}/best"
    print(f"Predictor: {pred_path}")
else:
    pred_path = f"/content/drive/MyDrive/kronos-th/checkpoints/{MODEL_NAME}/fold{FOLD}/best"
    print(f"Holdout eval using: {pred_path}")


In [ ]:
results = evaluate_model(
    checkpoint_path=pred_path,
    test_dataset=dataset['test'],
    max_samples=100,
)
print(f"Samples:              {results['n_samples']}")
print(f"Zero-shot hit-rate:   {results['zero_shot_hit_rate']:.1%}")
print(f"Fine-tuned hit-rate:  {results['fine_tuned_hit_rate']:.1%}")
print(f"Improvement:          {results['improvement_pp']:+.1%}")
print(f"Zero-shot MAE:        {results['zero_shot_mae']:.4f}")
print(f"Fine-tuned MAE:       {results['fine_tuned_mae']:.4f}")

In [ ]:
if MODE == "train":
    import shutil
    drive_path = f'/content/drive/MyDrive/kronos-th/checkpoints/{MODEL_NAME}/fold{FOLD}'
    shutil.copytree(pred_path, drive_path, dirs_exist_ok=True)
    print(f"Checkpoint saved to: {drive_path}")
else:
    print("Holdout mode — checkpoint not modified")